In [1]:
from transformers import T5TokenizerFast

In [2]:
tokenizer = T5TokenizerFast.from_pretrained("google-t5/t5-small")

/Users/Shaz/DS-GA 1011 NLP/nyu-dsga1011-a4/part2/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [3]:
def load_lines(path):
    with open(path, 'r') as f:
        lines = f.readlines()
        lines = [line.strip() for line in lines]
    return lines

In [7]:
train_nl = load_lines('data/train.nl')
train_sql = load_lines('data/train.sql')
dev_nl = load_lines('data/dev.nl')
dev_sql = load_lines('data/dev.sql')
len(train_nl),len(dev_nl)

(4225, 466)

In [8]:
def compute_stats(lines):
    lengths = []
    vocab = set()
    for line in lines:
        ids = tokenizer(line, add_special_tokens=False).input_ids  # skip the </s> token
        lengths.append(len(ids))
        vocab.update(ids)
    mean_length = sum(lengths) / len(lengths)
    return mean_length, len(vocab)

In [9]:
train_nl_len, train_nl_vocab = compute_stats(train_nl)
train_sql_len, train_sql_vocab = compute_stats(train_sql)
dev_nl_len, dev_nl_vocab = compute_stats(dev_nl)
dev_sql_len, dev_sql_vocab = compute_stats(dev_sql)

print(f"{'Statistic':<30}{'Train':>10}{'Dev':>10}")
print(f"{'Number of examples':<30}{len(train_nl):>10}{len(dev_nl):>10}")
print(f"{'Mean sentence length':<30}{train_nl_len:>10.2f}{dev_nl_len:>10.2f}")
print(f"{'Mean SQL query length':<30}{train_sql_len:>10.2f}{dev_sql_len:>10.2f}")
print(f"{'Vocabulary size (NL)':<30}{train_nl_vocab:>10}{dev_nl_vocab:>10}")
print(f"{'Vocabulary size (SQL)':<30}{train_sql_vocab:>10}{dev_sql_vocab:>10}")

Statistic                          Train       Dev
Number of examples                  4225       466
Mean sentence length               17.10     17.07
Mean SQL query length             216.37    210.05
Vocabulary size (NL)                 791       465
Vocabulary size (SQL)                555       395


In [ ]:
def preprocess_sql(sql):
    return sql.replace("<", "LESS")

def postprocess_sql(sql):
    return sql.replace("LESS", "<")


In [11]:
train_sql_pre = [preprocess_sql(s) for s in train_sql]
dev_sql_pre = [preprocess_sql(s) for s in dev_sql]

train_sql_len_pre, train_sql_vocab_pre = compute_stats(train_sql_pre)
dev_sql_len_pre, dev_sql_vocab_pre = compute_stats(dev_sql_pre)

def count_unk(lines):
    return sum(tokenizer.unk_token_id in tokenizer(line).input_ids for line in lines)

print(f"{'Statistic':<30}{'Train':>10}{'Dev':>10}")
print(f"{'Mean sentence length':<30}{train_nl_len:>10.2f}{dev_nl_len:>10.2f}")
print(f"{'Mean SQL query length':<30}{train_sql_len_pre:>10.2f}{dev_sql_len_pre:>10.2f}")
print(f"{'Vocabulary size (NL)':<30}{train_nl_vocab:>10}{dev_nl_vocab:>10}")
print(f"{'Vocabulary size (SQL)':<30}{train_sql_vocab_pre:>10}{dev_sql_vocab_pre:>10}")
print(f"{'SQL queries with <unk> (before)':<30}{count_unk(train_sql):>10}{count_unk(dev_sql):>10}")
print(f"{'SQL queries with <unk> (after)':<30}{count_unk(train_sql_pre):>10}{count_unk(dev_sql_pre):>10}")


Statistic                          Train       Dev
Mean sentence length               17.10     17.07
Mean SQL query length             216.37    210.05
Vocabulary size (NL)                 791       465
Vocabulary size (SQL)                555       395
SQL queries with <unk> (before)       393        45
SQL queries with <unk> (after)         0         0
